# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. 

Source: Original local package: 03_fastai/07_trees_and_random_forests.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '07_trees_and_random_forests.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


# Lesson 5: trees, split scores, bagging and random forests
Source task: https://github.com/fastai/course22/blob/master/07-how-random-forests-really-work.ipynb
This is an independently written solution companion, not a copy of the instructor notebook or a course completion record.


In [ ]:
import os,json
from pathlib import Path
import numpy as np,pandas as pd,torch
import matplotlib.pyplot as plt
torch.manual_seed(42);torch.set_num_threads(1)
np.random.seed(42)
SMOKE=os.environ.get('COURSE_SMOKE','0')=='1'
results=Path('results');results.mkdir(exist_ok=True)

from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from fastai_utils import *
frame,tr,va,processor,features,target=split_and_prepare()
def split_score(column,y,threshold):
    left=column<=threshold
    if left.all() or (~left).all():return float('inf')
    return float((left.sum()*np.std(y[left])+(~left).sum()*np.std(y[~left]))/len(y))
def best_split(column,y):
    values=np.unique(column);thresholds=(values[:-1]+values[1:])/2
    scored=[(split_score(column,y,t),float(t)) for t in thresholds]
    return min(scored) if scored else (float('inf'),None)
print('Best split per processed feature:',[best_split(features[tr,j],target[tr]) for j in range(features.shape[1])])
shallow=DecisionTreeClassifier(max_leaf_nodes=4,random_state=42).fit(features[tr],target[tr])
plt.figure(figsize=(11,5));plot_tree(shallow,filled=True,feature_names=processor.get_feature_names_out(),class_names=['died','survived']);plt.show()
def gini(labels):
    p=np.mean(labels);return 1-p*p-(1-p)*(1-p)
print('Training Gini:',gini(target[tr]))
rng=np.random.default_rng(42);trees=[]
for seed in range(50):
    bootstrap=rng.choice(tr,size=len(tr),replace=True)
    trees.append(DecisionTreeClassifier(min_samples_leaf=5,random_state=seed).fit(features[bootstrap],target[bootstrap]))
bagged=np.mean([tree.predict_proba(features[va])[:,1] for tree in trees],axis=0)
forest=RandomForestClassifier(n_estimators=100,min_samples_leaf=5,random_state=42,n_jobs=1,oob_score=True).fit(features[tr],target[tr])
report=pd.DataFrame([
 {'model':'shallow tree','validation_accuracy':accuracy_score(target[va],shallow.predict(features[va]))},
 {'model':'manual bagging','validation_accuracy':accuracy_score(target[va],bagged>=.5)},
 {'model':'random forest','validation_accuracy':accuracy_score(target[va],forest.predict(features[va]))}])
print(report.to_string(index=False));print('Forest OOB accuracy:',forest.oob_score_)
report.to_csv(results/'titanic_tree_comparison.csv',index=False)
pd.DataFrame({'feature':processor.get_feature_names_out(),'importance':forest.feature_importances_}).sort_values('importance',ascending=False).to_csv(results/'titanic_feature_importance.csv',index=False)

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)